# Crystalline: Qwen3.6-35B-A3B Research Pipeline

**Goal:** Analyze Qwen3.6's Gated DeltaNet architecture and design tropical adaptations.

**Note:** The full 35B model does not fit on Colab T4. This notebook:
1. Downloads Qwen3.6 config/tokenizer only
2. Analyzes architecture (MoE, DeltaNet, context length)
3. Simulates VRAM with expert offloading
4. Proposes a CrystallineMoEModel architecture
5. Generates synthetic data using a smaller teacher (Qwen2.5-0.5B)
6. Trains a Crystalline student with MoE + tropical DeltaNet blocks
7. Benchmarks and saves telemetry to Drive

## Section 1: Setup

In [ ]:
# ============================================================
# Section 1: Setup
# ============================================================
!pip install -q transformers accelerate bitsandbytes torch psutil tqdm huggingface_hub datasets matplotlib pandas numpy

import os, sys, json, time, math
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import Optional, List

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

# Mount Drive
from google.colab import drive
DRIVE_PATH = "/content/drive/MyDrive/CrystallineCache"
os.makedirs(DRIVE_PATH, exist_ok=True)
drive.mount("/content/drive", force_remount=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## Section 2: Qwen3.6 Architecture Analysis

In [ ]:
# ============================================================
# Section 2: Qwen3.6 Config Analysis
# ============================================================
MOE_MODEL = "Qwen/Qwen3.6-35B-A3B"

# Download only config and tokenizer (not weights)
from huggingface_hub import snapshot_download
MOE_CACHE = os.path.join(DRIVE_PATH, "models", MOE_MODEL.replace("/", "_"))
os.makedirs(MOE_CACHE, exist_ok=True)

if not os.path.exists(os.path.join(MOE_CACHE, "config.json")):
    print("Downloading Qwen3.6 config/tokenizer...")
    snapshot_download(
        repo_id=MOE_MODEL,
        local_dir=MOE_CACHE,
        local_dir_use_symlinks=False,
        allow_patterns=["config.json", "tokenizer.json", "tokenizer_config.json", "preprocessor_config.json"],
    )
    print("Downloaded.")
else:
    print("Config already cached.")

cfg = AutoConfig.from_pretrained(MOE_CACHE, trust_remote_code=True)
print("\n--- Qwen3.6-35B-A3B Architecture ---")
print(f"Hidden size: {cfg.hidden_size}")
print(f"Num layers: {cfg.num_hidden_layers}")
print(f"Num experts: {getattr(cfg, 'num_experts', getattr(cfg, 'num_local_experts', 'N/A'))}")
print(f"Active experts: {getattr(cfg, 'num_experts_per_tok', getattr(cfg, 'num_active_experts', 8))}")
print(f"Vocab size: {cfg.vocab_size}")
print(f"Context length: {getattr(cfg, 'max_position_embeddings', getattr(cfg, 'max_sequence_length', 32768))}")

## Section 3: MoE VRAM Simulation

In [ ]:
# ============================================================
# Section 3: MoE VRAM Simulation
# ============================================================
def simulate_moe_vram(
    total_params=35_000_000_000,
    active_params=3_000_000_000,
    num_experts=256,
    active_experts=8,
    bits_per_param=4,
    kv_cache_tokens=8192,
    kv_bits=4,
    hidden_size=3584,
    num_layers=40,
):
    bytes_per_param = bits_per_param / 8
    total_weight_gb = total_params * bytes_per_param / 1e9
    active_weight_gb = active_params * bytes_per_param / 1e9
    kv_gb = 2 * num_layers * kv_cache_tokens * hidden_size * (kv_bits / 8) / 1e9
    act_gb = 0.5
    gpu_total = active_weight_gb + kv_gb + act_gb
    offloaded_gb = total_weight_gb - active_weight_gb
    return {
        'total_weight_gb': total_weight_gb,
        'active_weight_gb': active_weight_gb,
        'kv_cache_gb': kv_gb,
        'activation_gb': act_gb,
        'gpu_total_gb': gpu_total,
        'offloaded_gb': offloaded_gb,
        'fits_t4': gpu_total < 16,
        'fits_a100': gpu_total < 40,
    }

scenarios = [
    {'name': '35B-A3B @ 4-bit (all experts)', 'bits': 4, 'offload': False},
    {'name': '35B-A3B @ 4-bit (expert offloading)', 'bits': 4, 'offload': True},
    {'name': '35B-A3B @ 3-bit (expert offloading)', 'bits': 3, 'offload': True},
]

print('\n' + '='*70)
print('MoE VRAM Simulation for Qwen3.6-35B-A3B')
print('='*70)
for s in scenarios:
    sim = simulate_moe_vram(bits_per_param=s['bits'])
    if s['offload']:
        gpu = sim['active_weight_gb'] + sim['kv_cache_gb'] + sim['activation_gb']
    else:
        gpu = sim['gpu_total_gb']
    print(f"\n{s['name']}:")
    print(f"  GPU VRAM:  {gpu:.2f} GB")
    print(f"  CPU RAM:   {sim['offloaded_gb']:.2f} GB (offloaded)")
    print(f"  KV cache:  {sim['kv_cache_gb']:.2f} GB")
    print(f"  Fits T4:   {gpu < 16}")
    print(f"  Fits A100: {gpu < 40}")
print('='*70)

## Section 4: Build CrystallineMoEModel

In [ ]:
# ============================================================
# Section 4: CrystallineMoEModel with Tropical DeltaNet
# ============================================================
import sys
sys.path.insert(0, '/content')  # adjust if repo is cloned elsewhere

try:
    from crystalline import CrystallineModel, CrystallineConfig
    from crystalline.train import train_crystalline_model, TextDataset, generate_synthetic_data
    from qwen_optimizer.benchmark import BenchmarkSuite
    from qwen_optimizer.telemetry import TelemetryLogger, TelemetryEntry
    print("Crystalline imported.")
except ImportError as e:
    print(f"Import error: {e}")
    print("Please upload the crystalline package or clone the repo.")
    raise

# Student config inspired by Qwen3.6 but much smaller for Colab
student_config = CrystallineConfig(
    vocab_size=cfg.vocab_size,
    d_model=512,
    num_layers=6,
    num_heads=8,
    d_ff=1024,
    max_seq_len=2048,
    dropout=0.1,
    use_delta_net=True,   # Enable tropical DeltaNet blocks
    num_experts=8,        # Small MoE for experimentation
    top_k=2,
)

moe_student = CrystallineModel(student_config).to(DEVICE)
print(f"CrystallineMoEModel created.")
print(f"Parameters: {sum(p.numel() for p in moe_student.parameters())/1e6:.1f}M")

## Section 5: Load Small Teacher (Qwen2.5-0.5B) for Synthetic Data

In [ ]:
# ============================================================
# Section 5: Load Qwen2.5-0.5B as Teacher
# ============================================================
from qwen_optimizer.download import ModelCache

TEACHER_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
cache = ModelCache(os.path.join(DRIVE_PATH, "models"))
teacher_path = cache.get_model_path(TEACHER_NAME)

print(f"Loading teacher: {TEACHER_NAME}")
teacher = AutoModelForCausalLM.from_pretrained(
    teacher_path,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else "cpu",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(teacher_path, trust_remote_code=True)
print("Teacher loaded.")

## Section 6: Distill into CrystallineMoEModel

In [ ]:
# ============================================================
# Section 6: Distillation Training
# ============================================================
OUTPUT_DIR = os.path.join(DRIVE_PATH, "qwen36_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Generating synthetic data...")
texts = generate_synthetic_data(
    teacher=teacher,
    tokenizer=tokenizer,
    num_samples=20,
    max_length=64,
    device=DEVICE,
)
dataset = TextDataset(texts, tokenizer, max_length=64)
print(f"Dataset: {len(dataset)} samples")

print("\nStarting distillation...")
trained_moe = train_crystalline_model(
    teacher=teacher,
    student=moe_student,
    tokenizer=tokenizer,
    dataset=dataset,
    epochs=1,
    batch_size=2,
    lr=5e-5,
    device=DEVICE,
    temperature=2.0,
    alpha=0.5,
    crystallization_weight=0.005,
    output_dir=OUTPUT_DIR,
    max_length=64,
)
print("Distillation complete.")

## Section 7: Crystallize + Benchmark

In [ ]:
# ============================================================
# Section 7: Crystallize and Benchmark
# ============================================================
bench = BenchmarkSuite(tokenizer, device=DEVICE)
logger = TelemetryLogger(os.path.join(DRIVE_PATH, "telemetry_qwen36.json"))

# Baseline teacher benchmark
base_result = bench.run_inference_benchmark(teacher, "teacher_baseline", "fp16")
logger.log(TelemetryEntry(
    timestamp=datetime.utcnow().isoformat(),
    stage=base_result.stage,
    model_name=TEACHER_NAME,
    quantization=base_result.quantization,
    vram_mb=base_result.vram_mb,
    tokens_per_sec_prefill=base_result.tokens_per_sec_prefill,
    tokens_per_sec_decode=base_result.tokens_per_sec_decode,
    perplexity=None,
    latency_ttft_ms=base_result.latency_ttft_ms,
    latency_tpot_ms=base_result.latency_tpot_ms,
    notes="teacher baseline",
))

# Distilled student
dist_result = bench.run_inference_benchmark(trained_moe, "distilled_moe", "crystalline_fp16")
logger.log(TelemetryEntry(
    timestamp=datetime.utcnow().isoformat(),
    stage=dist_result.stage,
    model_name="CrystallineMoE",
    quantization=dist_result.quantization,
    vram_mb=dist_result.vram_mb,
    tokens_per_sec_prefill=dist_result.tokens_per_sec_prefill,
    tokens_per_sec_decode=dist_result.tokens_per_sec_decode,
    perplexity=None,
    latency_ttft_ms=dist_result.latency_ttft_ms,
    latency_tpot_ms=dist_result.latency_tpot_ms,
    notes="MoE + tropical DeltaNet",
))

# Crystallize
trained_moe.crystallize()
cryst_result = bench.run_inference_benchmark(trained_moe, "crystallized_moe", "ternary")
logger.log(TelemetryEntry(
    timestamp=datetime.utcnow().isoformat(),
    stage=cryst_result.stage,
    model_name="CrystallizedMoE",
    quantization=cryst_result.quantization,
    vram_mb=cryst_result.vram_mb,
    tokens_per_sec_prefill=cryst_result.tokens_per_sec_prefill,
    tokens_per_sec_decode=cryst_result.tokens_per_sec_decode,
    perplexity=None,
    latency_ttft_ms=cryst_result.latency_ttft_ms,
    latency_tpot_ms=cryst_result.latency_tpot_ms,
    notes="weights in {-1,0,1}",
))

# Save
torch.save(trained_moe.state_dict(), os.path.join(DRIVE_PATH, "crystallized_moe.pt"))
logger.plot_comparison(save_path=os.path.join(DRIVE_PATH, "qwen36_benchmark_chart.png"))
logger.summary()

## Section 8: Next Steps for Full Qwen3.6

To scale this to the real Qwen3.6-35B-A3B:
1. Use llama.cpp GGUF on CPU for teacher inference (35B model too large for T4 GPU)
2. Generate large synthetic dataset from teacher
3. Train CrystallineMoEModel with ~1B parameters (6-8 layers, 512-768 d_model)
4. Use layer-wise distillation if loading full teacher on GPU
5. Benchmark on Colab A100 (40GB VRAM) for realistic comparison